# Baseline: TF-IDF + Logistic Regression

Three feature sets with the same classifier, compared on the validation set (overall and on the Z / Q / K / W slices). The test set is not touched here: it is used once, for the final model.

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import FeatureUnion, Pipeline

pd.set_option("display.max_colwidth", 120)

In [ ]:
# The MLflow server must be running in a separate terminal (see README)
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("support-triage")

In [ ]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")

X_train, y_train = train["text"], train["intent"]
X_val, y_val = val["text"], val["intent"]
len(train), len(val)

## Feature sets

- **word**: word unigrams + bigrams
- **char**: character 2–5-grams inside word boundaries (robust to typos)
- **word+char**: both, concatenated

In [ ]:
def word_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)


def char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)


feature_sets = {
    "word": word_tfidf(),
    "char": char_tfidf(),
    "word+char": FeatureUnion([("word", word_tfidf()), ("char", char_tfidf())]),
}
SLICES = ["Z", "Q", "K", "W"]  # typos, colloquial, keyword mode, offensive

## Training

One MLflow run per feature set: parameters, validation metrics (overall and per slice) and the fitted pipeline.

In [ ]:
def evaluate(model, X, y, flags) -> dict:
    pred = model.predict(X)
    metrics = {
        "macro_f1": f1_score(y, pred, average="macro"),
        "accuracy": accuracy_score(y, pred),
    }
    for tag in SLICES:
        mask = flags.str.contains(tag).to_numpy()
        metrics[f"f1_{tag}"] = f1_score(y[mask], pred[mask], average="macro")
    return metrics


results, models = {}, {}
for name, features in feature_sets.items():
    with mlflow.start_run(run_name=f"baseline_logreg_{name}"):
        model = Pipeline([("features", features), ("clf", LogisticRegression(max_iter=2000))])
        model.fit(X_train, y_train)
        metrics = evaluate(model, X_val, y_val, val["flags"])

        mlflow.set_tag("stage", "baseline")
        mlflow.log_params(
            {"model": "logreg", "features": name, "C": 1.0, "train_rows": len(train)}
        )
        mlflow.log_metrics({f"val_{k}": v for k, v in metrics.items()})
        mlflow.sklearn.log_model(model, name="model")

    results[name], models[name] = metrics, model

results = pd.DataFrame(results).T.rename_axis("features").round(4)
results

## Error analysis

Most frequent confusions of the best feature set on the validation set.

In [ ]:
best_name = results["macro_f1"].idxmax()
pred = models[best_name].predict(X_val)

labels = sorted(y_val.unique())
cm = pd.DataFrame(confusion_matrix(y_val, pred, labels=labels), index=labels, columns=labels)
cm.index.name, cm.columns.name = "true", "pred"

pairs = cm.stack().rename("count").reset_index()
print(f"Best feature set: {best_name}")
errors_only = (pairs["true"] != pairs["pred"]) & (pairs["count"] > 0)
pairs[errors_only].sort_values("count", ascending=False).head(10)

In [ ]:
errors = val.assign(pred=pred).query("intent != pred")
errors[["instruction", "intent", "pred", "flags"]].sample(min(15, len(errors)), random_state=42)